In [ ]:
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import matplotlib.pyplot as plt
import Plot_setting
import numpy as np
from scipy.special import jv

INF = np.inf

hbar = 1
e = 1
#beta = INF  # Zero-temperature case: T = 0, beta -> inf
phi_quantum = 1 / 2

# Circuit parameters
omega_a = 25.338776456203686
omega_b = 2 * omega_a

phi_a = 0.11
phi_b = 0.204

E_J = 37.12 * 2 * np.pi 

# Environment
kappa_b = 4 / 10.4 # 1 / 10.4  # 1/ns from Leon's paper

omega_d = omega_b                  # Drive frequency
omega_p = 2 * omega_a - omega_d   # Pump frequency

epsilon_p = 0.1
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b  # Leon's paper
epsilon_d = 4 * g2

g = E_J * np.sin(epsilon_p) * phi_a**2 * phi_b
epsilon_d_ratio = 2


A_0 = np.sqrt(epsilon_d / g)
B_0 = - np.sqrt(2) * epsilon_d / kappa_b

# Solution for $\kappa_a = 0$

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# parameters
c = np.sqrt(2) * g

# reduced vector field X = (A, PB)
def F_no_loss_a(X):
    A, PA, B, PB = X
    dPA =  - c * A_0 * B
    dA = -dPA
    dB = np.sqrt(2) * g * PA * (A_0 - PA) - 0.5 * kappa_b * B
    dPB = -dB
    return np.array([dA, dPA, dB, dPB])

def tangent(string):
    T = np.zeros_like(string)
    T[1:-1] = string[2:] - string[:-2]
    T[0] = string[1] - string[0]
    T[-1] = string[-1] - string[-2]
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return T / norms

def reparametrize(string):
    # arc-length parametrization
    ds = np.linalg.norm(np.diff(string, axis=0), axis=1)
    s = np.zeros(len(string))
    s[1:] = np.cumsum(ds)

    if s[-1] == 0:
        return string.copy()

    s_uniform = np.linspace(0, s[-1], len(string))

    A_interp = interp1d(s, string[:, 0], kind='linear')
    PA_interp = interp1d(s, string[:, 1], kind='linear')
    B_interp = interp1d(s, string[:, 2], kind='linear')
    PB_interp = interp1d(s, string[:, 3], kind='linear')

    new_string = np.zeros_like(string)
    new_string[:, 0] = A_interp(s_uniform)
    new_string[:, 1] = PA_interp(s_uniform)
    new_string[:, 2] = B_interp(s_uniform)
    new_string[:, 3] = PB_interp(s_uniform)
    return new_string

def initialize_string_adiabatic(N):
    # better than a straight line: use adiabatic curve
    PA_vals = np.linspace(0.0, A_0, N)
    A_vals = A_0 - PA_vals
    B_vals = (2*np.sqrt(2)*g/kappa_b) * PA_vals * (A_0 - PA_vals)
    PB_vals = -B_vals
    return np.column_stack([A_vals, PA_vals, B_vals, PB_vals])

def string_step(string, F, dtau, init_point, end_point):
    new_string = string.copy()
    T = tangent(string)

    for i in range(1, len(string)-1):
        Fi = F(string[i])

        # normal component only
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]

        new_string[i] += dtau * Fi_normal

    # fix endpoints
    new_string[0] = init_point
    new_string[-1] = end_point

    # redistribute images
    new_string = reparametrize(new_string)

    # fix endpoints again after interpolation
    new_string[0] = init_point
    new_string[-1] = end_point

    return new_string

def string_method(initialize_string, F, N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    string = initialize_string(N)
    init_point = string[0]
    end_point = string[-1]
    err_list = []
    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, F, dtau, init_point, end_point)

        err = np.max(np.linalg.norm(string - old, axis=1))
        err_list.append(err)
        if k % 200 == 0:
            print(f"iter={k}, err={err:.3e}")

        if err < tol:
            print(f"converged at iter={k}, err={err:.3e}")
            break
    return string, err_list

def string_method_bis(initialize_string, F, N=200, dtau=1e-3, max_iter=20000, tol=1e-8,
                   dtau_min=1e-10, dtau_max=1e-2, shrink=0.5, grow=1.02,
                   min_rejections=50, verbose_every=5):
    string = initialize_string(N)
    init_point = string[0]
    end_point = string[-1]
    err_prev = INF
    rejections_in_a_row = 0
    k = 0
    err_list = []

    while k < max_iter:
        trial = string_step(string, F, dtau, init_point, end_point)
        err = np.max(np.linalg.norm(trial - string, axis=1))
        err_list.append(err)

        # reject the step if it made things worse, and shrink dtau
        if err > err_prev and dtau > dtau_min and rejections_in_a_row > min_rejections:
            dtau = max(dtau * shrink, dtau_min)
            rejections_in_a_row = 0
            continue  # retry from the same string, don't advance k

        # accept the step
        rejections_in_a_row +=1
        string = trial

        if k % verbose_every == 0:
            print(f"iter={k}, err={err:.3e}, dtau={dtau:.2e}")

        if err < tol:
            print(f"converged at iter={k}, err={err:.3e}, dtau={dtau:.2e}")
            break

        err_prev = err
        dtau = min(dtau * grow, dtau_max)  # ease back up when things are stable
        k += 1
    else:
        print(f"did not converge within {max_iter} iterations (err={err:.3e}, dtau={dtau:.2e})")

    return string, err_list

In [ ]:
N=200
string_init = initialize_string_adiabatic(N)
string_test, err_list = string_method(initialize_string_adiabatic, F_no_loss_a, dtau=1e-2, max_iter=30000)
string_test_bis, err_list_bis = string_method_bis(initialize_string_adiabatic, F_no_loss_a, dtau=1e-2, max_iter=30000)

In [ ]:
plt.plot(err_list[1:])
plt.plot(err_list_bis[1:], ls="--")
#plt.ylim(8.32e-4, 8.42e-4)
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Error")
plt.show()

In [ ]:
fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.plot(string_test[:,1],string_test[:,2])
ax_1.plot(string_init[:,1],string_init[:,2])
ax_1.set_xlabel(r'$P_A$')
ax_1.set_ylabel(r'$B$')

ax_2.plot(string_test[:,0],string_test[:,3])
ax_2.plot(string_init[:,0],string_init[:,3])
ax_2.set_xlabel(r'$A$')
ax_2.set_ylabel(r'$P_B$')

plt.tight_layout()
plt.show()

# Let's increase slowly $\kappa_a$

In [ ]:
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2

delta = kappa_a * kappa_b / 4 / g**2
a = (A_0**2)/(2*A_0**2 - 2*delta)*(1 + (2*delta)/(A_0**2)-np.sqrt(1+4*(2*delta)/(A_0**2)))
P_A_value = np.sqrt(delta/(a*(a**2-1)))
A_value = a*P_A_value
B_value = -2*np.sqrt(2)*g/kappa_b*A_value**2-kappa_a/(2*np.sqrt(2)*g)
P_B_value = -2*np.sqrt(2)*g*A_value*P_A_value/kappa_b

In [ ]:
def F_loss_a(X):
    A, PA, B, PB = X
    dA = c*(A*B - PA*PB) + kappa_a * (A/2+PA)
    dPA =   c * (A * PB - PA * B) - kappa_a * PA/2
    dB =  epsilon_d/np.sqrt(2) - g*(A**2 + PA**2)/np.sqrt(2) + kappa_b * (B/2 + PB)
    dPB = - c * A * PA - kappa_b * PB/2
    return np.array([dA, dPA, dB, dPB])

def initialize_string(N):
    # better than a straight line: use adiabatic curve
    PA_vals = np.linspace(0.0, P_A_value, N)
    A_vals = np.linspace(np.sqrt(A_0**2-delta), A_value, N)
    B_vals = -np.sqrt(2)*epsilon_d/kappa_b + np.sqrt(2)*g*(A_vals**2 + PA_vals**2)/kappa_b + 4*np.sqrt(2)*g*A_vals*PA_vals/kappa_b
    PB_vals = -2*np.sqrt(2)*g*A_vals*PA_vals/kappa_b
    return np.column_stack([A_vals, PA_vals, B_vals, PB_vals])


def string_method(initialize_string, F, N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    string = initialize_string(N)
    init_point = string[0]
    end_point = string[-1]
    err_list = []
    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, F, dtau, init_point, end_point)

        err = np.max(np.linalg.norm(string - old, axis=1))
        err_list.append(err)
        if k % 200 == 0:
            print(f"iter={k}, err={err:.3e}")

        if err < tol:
            print(f"converged at iter={k}, err={err:.3e}")
            break
    return string, err_list

In [ ]:
N=200
string_init = initialize_string(N)
string_test_loss, err_list = string_method(initialize_string, F_loss_a, dtau=1e-4, max_iter=60_000)

In [ ]:
plt.plot(err_list[1:])
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Error")
plt.show()

In [ ]:
fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.plot(string_test_loss[:,1],string_test_loss[:,2])
ax_1.plot(string_init[:,1],string_init[:,2])
ax_1.set_xlabel(r'$P_A$')
ax_1.set_ylabel(r'$B$')

ax_2.plot(string_test_loss[:,0],string_test_loss[:,3])
ax_2.plot(string_init[:,0],string_init[:,3])
ax_2.set_xlabel(r'$A$')
ax_2.set_ylabel(r'$P_B$')

plt.tight_layout()
plt.show()